In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import linear_model
import DataCleaning

df_clean = DataCleaning.df_clean

In [ ]:
df_clean["InvoiceMonth"] = pd.to_datetime(df_clean["InvoiceDate"]).dt.month #create a month variable for the linear Regression

# variables for the posttaxprofit
df_clean["PostTaxProfitIfNexus"] = df_clean["Profit"]-df_clean["SalesTaxNoNexus"] 
df_clean["ActualPostTaxProfit"] = df_clean["Profit"] - df_clean["SalesTaxOwed"]
# group dataframe for linear regression
df_grouped = df_clean.groupby(by=["State", "InvoiceYear", "InvoiceMonth"])[["ActualPostTaxProfit", "PostTaxProfitIfNexus"]].sum().reset_index()
df_grouped
growth_trends = []

# loop to predict the profit for all states
for state in df_grouped["State"].unique():
    df_state = df_grouped.loc[df_grouped["State"] == state].copy()
    # add the time index
    df_state["time"] = range(len(df_state))
    
    #predict profit for the actual profit
    model_actual = linear_model.LinearRegression()
    model_actual.fit(df_state[["time"]], df_state[["ActualPostTaxProfit"]])
    actual_post_tax_profit_growth = model_actual.coef_[0][0]
    
    #predict profit for the profitif there was a nexus
    model_ifnexus = linear_model.LinearRegression()
    model_ifnexus.fit(df_state[["time"]], df_state[["PostTaxProfitIfNexus"]])
    post_tax_profit_ifnexus_growth = model_ifnexus.coef_[0][0]

    # prediction for in 6 months
    six_month_prediction = len(df_state)+5

    # save and append the predictions and the growth in the list
    growth_trends.append({
        "State": state,
        "ActualPostTaxProfit": model_actual.predict([[six_month_prediction]])[0][0],
        "ActualPostTaxProfitGrowth": actual_post_tax_profit_growth,
        "PostTaxProfitIfNexus": model_ifnexus.predict([[six_month_prediction]])[0][0],
        "PostTaxProfitIfNexusGrowth": post_tax_profit_ifnexus_growth,
    })


# put list into dataframe
df_trends = pd.DataFrame(growth_trends)
df_trends = df_trends.sort_values(by=("ActualPostTaxProfit"), ascending=False)
df_trends

#create a multiple bar diagram to compare the profits 
df_trends.plot(kind="bar",x="State", y=["ActualPostTaxProfit", "PostTaxProfitIfNexus"])

plt.xlabel("State", fontsize =12)
plt.ylabel("Profit after Tax", fontsize = 12)
plt.title("Comparison between the forecasted actual profit and the profit if there was a nexus", fontsize =15)

                                             


